In [1]:
from pathlib import Path

import prism

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
    RestOf
)
from imagematerials.preprocessing import get_preprocessing_data
import numpy as np

from imagematerials.rest_of import rest_of_preprocessing
import matplotlib.pyplot as plt

from imagematerials.electricity.preprocessing import (
    get_preprocessing_data_gen,
    get_preprocessing_data_grid)


In [2]:
scenario_name = "SSP2_M_CP"

In [3]:
climate_policy_scenario_dir = Path("..", "data", "raw", "image", scenario_name)
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

path_image_materials = Path("C:/Coding/image-materials")

raw_data = path_image_materials / "data/raw"

In [4]:


# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2100, 1)
simulation_timeline = prism.Timeline(1970, 2100, 1)


bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), 
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None) 
vhc_sector = get_preprocessing_data("vehicles", raw_data, 
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
rest_sector = rest_of_preprocessing(raw_data, 
                    image_scenario_directory = climate_policy_scenario_dir, 
                    scenario = scenario_name)

# needed for electricity model
scen = scenario_name.split("_")[0]
variant = "_".join(scenario_name.split("_")[1:])

# TODO fix this for real in the future
prep_data_vhc = vhc_sector.prep_data
prep_data_el = get_preprocessing_data_gen(path_image_materials, scen, variant,
                                          1971, 2100, 2100)
_, prep_data_grid = get_preprocessing_data_grid(path_image_materials, scen, variant,
                                                1971, 2100, 2100)


vhc_sector = Sector('vehicles', prep_data_vhc)
rest_sector = Sector(name='rest_of', 
                    data = rest_sector,)
sec_electr_gen = Sector("generation", prep_data_el)
sec_electr_grid = Sector("grid", prep_data_grid)

factory = ModelFactory(
[bld_sector, vhc_sector, rest_sector, sec_electr_gen, sec_electr_grid], complete_timeline
).add(GenericStocks, ["buildings", "vehicles", "generation", "grid"]
).add(GenericMaterials,  "vehicles"
).add(MaterialIntensities, ["buildings", "generation", "grid"]
).add(RestOf, "rest_of", input_sources={
    "gompertz_coefs": "rest_of",
    "gdp_per_capita": "rest_of",
    "population": "rest_of",
    "historic_diff_consumption_mean": "rest_of",
    "historic_diff_consumption_total": "rest_of"
}
)
model = factory.finish()

import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)



C:\Coding\image-materials\imagematerials\buildings\preprocessing\main.py:75: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'material' ('material',) The recommendation is to set join explicitly for this case.
  mat_intensities = xr.concat((mat_intensities_res, mat_intensities_comm), dim="Type")


Path to image output: C:\Coding\image-materials\data\raw\image\SSP2_M_CP\EnergyServices
gcap_stock to xarray Dataset
gcap_types_materials to xarray Dataset
gcap_lifetime_distr not in conversion_table
grid_stock_lines to xarray Dataset
materials_grid_kgperkm to xarray Dataset
lifetime_grid_distr not in conversion_table
grid_stock_add to xarray Dataset
materials_grid_add_kgperunit to xarray Dataset
lifetime_grid_distr not in conversion_table


In [ ]:
# save totals of materials for rest-of fitting

from imagematerials.rest_of.preprocessing.sum_sectors_image_materials import sum_inflows_for_output

materials_dict_metal = {
    'Steel' : 'Steel',
    'Aluminium' : 'Aluminium',
    'Copper' : 'Cu',
}

materials_dict_nmm = {
    'Cement' : 'Cement',
    'Sand' : 'Sand'
}

materials_dict_sand = {
    'Sand' : 'Sand'
}
total_material_metals = sum_inflows_for_output(model, materials_dict_metal, 'metals')
total_material_nmm = sum_inflows_for_output(model, materials_dict_nmm, 'nmm')
total_material_sand = sum_inflows_for_output(model, materials_dict_sand, 'nmm')

Steel
done steel
Aluminium
done aluminium
Copper


c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


done copper
Cement
done cement
Sand
done sand_gravel_crushed_rock
Sand
done sand_gravel_crushed_rock


c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


In [11]:
total_material_sand

{'sand_gravel_crushed_rock': Region      class_ 1      class_ 2      class_ 3      class_ 4      class_ 5  \
 time                                                                           
 1971    2.385189e+06  2.435285e+07  3.730027e+06  1.705462e+06  3.458665e+06   
 1972    3.467571e+06  3.710797e+07  5.322846e+06  1.644901e+06  8.338012e+06   
 1973    2.376599e+06  3.586547e+07  5.270906e+06  1.513045e+06  9.609074e+06   
 1974    1.493432e+06  8.205950e+06  4.188471e+06  1.256864e+06  9.019989e+06   
 1975    3.215281e+06  8.983147e+06  4.584884e+06  1.337109e+06  4.222079e+06   
 ...              ...           ...           ...           ...           ...   
 2096    5.490954e+06  4.216694e+07  1.640038e+07  1.121471e+07  1.535040e+07   
 2097    5.468527e+06  4.209309e+07  1.637940e+07  1.101967e+07  1.542384e+07   
 2098    5.511280e+06  4.231812e+07  1.645529e+07  1.088543e+07  1.564502e+07   
 2099    5.517309e+06  4.259858e+07  1.628660e+07  1.088017e+07  1.539645e+07   
